# Municipality Count — Norte Amazonia Bolivia

This notebook processes the **2024 Bolivian census data** to estimate the number of infrastructure
units (schools, health centers, shops, etc.) required in each municipality of the northern Amazon region.

Two output datasets are produced:

| Output file | Description |
|---|---|
| `municipalities_counts.csv / .xlsx` | Based on **non-electrified** households only (1 900-person threshold) |
| `municipalities_counts_total.csv` | Based on **all** 2024 households — full infrastructure need |

**Census source**: INE, *Censo Bolivia 2024 — Datos y Estadísticas Clave* (2025)  
**Allocation rules**: Arias thesis methodology (Lowlands)

In [ ]:
import pandas as pd
import numpy as np

# ── Column name constants (long INE census headers) ─────────────────────────
COL_HH_GRID   = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Servicio público de energía eléctrica"
COL_HH_MOTOR  = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Motor propio"
COL_HH_SOLAR  = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Panel solar"
COL_HH_OTHER  = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Otra"
COL_HH_NO     = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | No tiene"
COL_PER_NO    = "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD | 2024 | No tiene"
COL_PER_TOTAL = "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD | 2024 | Total"

# ── Infrastructure allocation rules (Arias thesis, Lowlands methodology) ────
# Format: (output_column, units_per_n_households_or_persons, minimum_count)
INFRA_RULES_HH = [         # applied to household count
    ("public_lighting",        10,  0),
    ("store",                  30,  1),
    ("restaurant",             30,  1),
    ("workshop",               60,  1),
    ("entertainment_business", 60,  1),
    ("rice_processing",       200,  0),
]
INFRA_RULES_PER = [        # applied to person count
    ("school",        5000, 1),
    ("health_center", 5000, 1),
]
THRESHOLD_PERSONS = 1900   # min. non-electrified persons to justify a school/health center

## 1. Load census data

In [2]:
df = pd.read_csv("output/CSV_final.csv", sep=",")
print(f"Census data loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Census data loaded: 29 rows × 122 columns


## 2. Clean data and build municipality identifiers

Drop aggregate rows where `MUNICIPIO/TIOC` is empty (these are regional summary rows, not individual municipalities).

Then build a clean `municipality` identifier in snake_case, with a department suffix to distinguish the two cities named **Santa Rosa** (one in Beni, one in Pando).

In [3]:
# Drop aggregate rows (empty MUNICIPIO/TIOC = regional summary rows, not individual municipalities)
df_clean = df.dropna(subset=["MUNICIPIO/TIOC"]).copy()

def build_municipality_id(row):
    """Snake_case municipality name, with department suffix to disambiguate duplicate names."""
    muni = str(row["MUNICIPIO/TIOC"]).strip()
    dept = str(row["DEPARTAMENTO"]).strip()
    if muni == "Santa Rosa":          # two cities share this name (Beni and Pando)
        return f"Santa_Rosa_{dept}"
    return muni.replace(" ", "_")

df_clean["municipality"] = df_clean.apply(build_municipality_id, axis=1)
df_clean["department"]   = df_clean["DEPARTAMENTO"]

## 3. Infrastructure allocation — non-electrified households

For each municipality, estimate the number of infrastructure units based solely on
**non-electrified** households and persons (census 2024, *"No tiene"* electricity source).

The rules come from the **Arias thesis (Lowlands methodology)**:

| Infrastructure | Based on | Rule |
|---|---|---|
| School | Persons without electricity | 1 per 5 000 persons, if ≥ 1 900 persons |
| Health center | Persons without electricity | 1 per 5 000 persons, if ≥ 1 900 persons |
| Public lighting | Non-electrified households | 1 per 10 hh |
| Store | Non-electrified households | 1 per 30 hh (min 1) |
| Restaurant | Non-electrified households | 1 per 30 hh (min 1) |
| Workshop | Non-electrified households | 1 per 60 hh (min 1) |
| Entertainment business | Non-electrified households | 1 per 60 hh (min 1) |
| Rice processing | Non-electrified households | 1 per 200 hh |

<div style="background:#d1ecf1; border-left:5px solid #17a2b8; padding:10px; border-radius:4px; margin:8px 0">
ℹ️ The 1 900-person threshold prevents allocating schools or health centers to very small
isolated communities where a shared facility would be more appropriate.
</div>

In [4]:
df_clean["non_elec_hh"]     = df_clean[COL_HH_NO].fillna(0).astype(int)
df_clean["persons_no_elec"] = df_clean[COL_PER_NO].fillna(0).astype(int)

# Household-based allocations (vectorized)
for col, ratio, min_val in INFRA_RULES_HH:
    df_clean[col] = np.maximum(min_val, df_clean["non_elec_hh"] // ratio)

# Person-based allocations — apply threshold to skip very small communities
for col, ratio, min_val in INFRA_RULES_PER:
    counts = np.maximum(min_val, df_clean["persons_no_elec"] // ratio)
    df_clean[col] = np.where(df_clean["persons_no_elec"] >= THRESHOLD_PERSONS, counts, 0)

OUTPUT_COLS = [
    "department", "municipality", "non_elec_hh", "persons_no_elec",
    "school", "health_center", "public_lighting", "store", "restaurant",
    "workshop", "entertainment_business", "rice_processing",
]
df_nonelec = df_clean[OUTPUT_COLS].sort_values(["department", "municipality"])

numeric_cols = df_nonelec.select_dtypes(include="number").columns
total_row = df_nonelec[numeric_cols].sum().to_frame().T
total_row["department"]   = "Amazonia_Norte"
total_row["municipality"] = "TOTAL"

df_out_nonelec = pd.concat([df_nonelec, total_row], ignore_index=True)
df_out_nonelec

,department,municipality,non_elec_hh,persons_no_elec,school,health_center,public_lighting,store,restaurant,workshop,entertainment_business,rice_processing
0,Beni,Exaltación,273,1291,0,0,27,9,9,4,4,1
1,Beni,Guayaramerín,825,2373,1,1,82,27,27,13,13,4
2,Beni,Reyes,711,1991,1,1,71,23,23,11,11,3
3,Beni,Riberalta,2925,9103,1,1,292,97,97,48,48,14
4,Beni,Santa_Rosa_Beni,407,1583,0,0,40,13,13,6,6,2
5,La Paz,Ixiamas,1043,3480,1,1,104,34,34,17,17,5
6,Pando,Bella_Flor,351,751,0,0,35,11,11,5,5,1
7,Pando,Bolpebra,253,663,0,0,25,8,8,4,4,1
8,Pando,Cobija,395,1077,0,0,39,13,13,6,6,1
9,Pando,Filadelfia,594,1603,0,0,59,19,19,9,9,2


## 4. Export — non-electrified municipalities

Add a `TOTAL` summary row (sum of all numeric columns) and save to both CSV and Excel.

In [5]:
df_out_nonelec.to_csv("output/municipalities_counts.csv",    index=False)
df_out_nonelec.to_excel("output/municipalities_counts.xlsx", index=False)

print(f"Saved {len(df_out_nonelec) - 1} municipalities + TOTAL row.")

Saved 21 municipalities + TOTAL row.


## 5. Full population totals (all households)

A second dataset uses **all 2024 households** (regardless of electricity source) and **total population**, without the 1 900-person threshold. This represents the full infrastructure need for the municipality, not just the non-electrified fraction.

| Infrastructure | Based on | Rule |
|---|---|---|
| School | Total persons | 1 per 5 000 persons (min 1, no threshold) |
| Health center | Total persons | 1 per 5 000 persons (min 1, no threshold) |
| Public lighting | Total households | 1 per 10 hh |
| Store | Total households | 1 per 30 hh (min 1) |
| Restaurant | Total households | 1 per 30 hh (min 1) |
| Workshop | Total households | 1 per 60 hh (min 1) |
| Entertainment business | Total households | 1 per 60 hh (min 1) |
| Rice processing | Total households | 1 per 200 hh |

Saved to `output/municipalities_counts_total.csv`.

In [6]:
ALL_HH_COLS = [COL_HH_GRID, COL_HH_MOTOR, COL_HH_SOLAR, COL_HH_OTHER, COL_HH_NO]
df_clean["total_hh"]      = df_clean[ALL_HH_COLS].sum(axis=1).astype(int)
df_clean["total_persons"] = df_clean[COL_PER_TOTAL].fillna(0).astype(int)

df_total = df_clean.copy()

for col, ratio, min_val in INFRA_RULES_HH:
    df_total[col] = np.maximum(min_val, df_total["total_hh"] // ratio)

for col, ratio, min_val in INFRA_RULES_PER:
    df_total[col] = np.maximum(min_val, df_total["total_persons"] // ratio)

OUTPUT_COLS_TOTAL = [
    "department", "municipality", "total_hh", "total_persons",
    "school", "health_center", "public_lighting",
    "store", "restaurant", "workshop",
    "entertainment_business", "rice_processing",
]
df_total = df_total[OUTPUT_COLS_TOTAL].sort_values(["department", "municipality"])

numeric_cols_t = df_total.select_dtypes(include="number").columns
total_row_t = df_total[numeric_cols_t].sum().to_frame().T
total_row_t["department"]   = "Amazonia_Norte"
total_row_t["municipality"] = "TOTAL"

df_out_total = pd.concat([df_total, total_row_t], ignore_index=True)
df_out_total.to_csv("output/municipalities_counts_total.csv", index=False)

print(f"Saved {len(df_out_total) - 1} municipalities + TOTAL row.")
df_out_total.tail(5)

Saved 21 municipalities + TOTAL row.


,department,municipality,total_hh,total_persons,school,health_center,public_lighting,store,restaurant,workshop,entertainment_business,rice_processing
17,Pando,Santa_Rosa_Pando,860,2827,1,1,86,28,28,14,14,4
18,Pando,Santos_Mercado,601,2282,1,1,60,20,20,10,10,3
19,Pando,Sena,2875,10247,2,2,287,95,95,47,47,14
20,Pando,Villa_Nueva,708,2485,1,1,70,23,23,11,11,3
21,Amazonia_Norte,TOTAL,84209,317517,62,62,8412,2796,2796,1393,1393,413
